In [0]:
# Step 1: Load Gold scored GN-level telco opportunity table

gold_table = "workspace.census360.gold_gn_telco_scored_v2"

df = spark.table(gold_table)

display(df.limit(10))

In [0]:
# Validate row count and columns

print("Total rows:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()

In [0]:
# Step 2: Select ML feature columns and target column

feature_cols = [
    "population_total",
    "occupied_housing_units",
    "household_size",
    "male_share",
    "female_share",
    "child_share",
    "working_age_share",
    "elderly_share",
    "population_score",
    "housing_score",
    "household_size_score",
    "child_share_score",
    "working_age_score"
]

target_col = "telco_demand_opportunity_score"

ml_df = df.select(
    "gnd_uid",
    "province_name",
    "district_name",
    "ds_division_name",
    "gn_division_name",
    *feature_cols,
    target_col,
    "opportunity_tier"
)

display(ml_df.limit(10))

In [0]:
# Validate selected ML dataset

print("ML rows:", ml_df.count())
print("ML columns:", len(ml_df.columns))

ml_df.printSchema()

In [0]:
# Step 3: Check null values in ML dataset

from pyspark.sql.functions import col, sum as spark_sum, when

null_check = ml_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ml_df.columns
])

display(null_check)

In [0]:
# Fill null values in numeric ML feature columns and target column

numeric_cols = feature_cols + [target_col]

ml_clean_df = ml_df.fillna(0, subset=numeric_cols)

display(ml_clean_df.limit(10))

In [0]:
# Validate null values after cleaning

null_check_after = ml_clean_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in numeric_cols
])

display(null_check_after)

print("Clean ML rows:", ml_clean_df.count())

In [0]:
# Step 4: Create ML feature vector

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

ml_vector_df = assembler.transform(ml_clean_df)

display(
    ml_vector_df.select(
        "gnd_uid",
        "province_name",
        "district_name",
        "ds_division_name",
        "gn_division_name",
        "features",
        target_col,
        "opportunity_tier"
    ).limit(10)
)

In [0]:
# Validate vectorized ML dataset

print("Vectorized ML rows:", ml_vector_df.count())

ml_vector_df.select("features", target_col).printSchema()

In [0]:
# Step 5: Split data into training and testing datasets

train_df, test_df = ml_vector_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())

In [0]:
# Validate train and test schema

train_df.select("features", target_col).printSchema()
test_df.select("features", target_col).printSchema()

In [0]:
# Step 6: Train Linear Regression model

from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol=target_col,
    predictionCol="lr_predicted_opportunity_score"
)

lr_model = lr.fit(train_df)

print("Linear Regression model trained successfully.")

In [0]:
# Check Linear Regression model details

print("Intercept:", lr_model.intercept)
print("Number of coefficients:", len(lr_model.coefficients))
print("Coefficients:", lr_model.coefficients)

In [0]:
# Step 7: Evaluate Linear Regression model

from pyspark.ml.evaluation import RegressionEvaluator

lr_predictions = lr_model.transform(test_df)

display(
    lr_predictions.select(
        "gnd_uid",
        "province_name",
        "district_name",
        "ds_division_name",
        "gn_division_name",
        target_col,
        "lr_predicted_opportunity_score",
        "opportunity_tier"
    ).limit(20)
)

In [0]:
# Calculate Linear Regression evaluation metrics

rmse_evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="lr_predicted_opportunity_score",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="lr_predicted_opportunity_score",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="lr_predicted_opportunity_score",
    metricName="r2"
)

lr_rmse = rmse_evaluator.evaluate(lr_predictions)
lr_mae = mae_evaluator.evaluate(lr_predictions)
lr_r2 = r2_evaluator.evaluate(lr_predictions)

print("Linear Regression RMSE:", lr_rmse)
print("Linear Regression MAE:", lr_mae)
print("Linear Regression R2:", lr_r2)

In [0]:
# Step 8: Train Random Forest Regression model

from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=target_col,
    predictionCol="rf_predicted_opportunity_score",
    numTrees=50,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_df)

print("Random Forest Regression model trained successfully.")

In [0]:
# Check Random Forest model details

print("Number of trees:", rf_model.getNumTrees)
print("Feature importances:", rf_model.featureImportances)

In [0]:
# Step 9: Evaluate Random Forest Regression model

rf_predictions = rf_model.transform(test_df)

display(
    rf_predictions.select(
        "gnd_uid",
        "province_name",
        "district_name",
        "ds_division_name",
        "gn_division_name",
        target_col,
        "rf_predicted_opportunity_score",
        "opportunity_tier"
    ).limit(20)
)

In [0]:
# Calculate Random Forest evaluation metrics

rf_rmse_evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="rf_predicted_opportunity_score",
    metricName="rmse"
)

rf_mae_evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="rf_predicted_opportunity_score",
    metricName="mae"
)

rf_r2_evaluator = RegressionEvaluator(
    labelCol=target_col,
    predictionCol="rf_predicted_opportunity_score",
    metricName="r2"
)

rf_rmse = rf_rmse_evaluator.evaluate(rf_predictions)
rf_mae = rf_mae_evaluator.evaluate(rf_predictions)
rf_r2 = rf_r2_evaluator.evaluate(rf_predictions)

print("Random Forest RMSE:", rf_rmse)
print("Random Forest MAE:", rf_mae)
print("Random Forest R2:", rf_r2)

In [0]:
# Step 10: Compare Linear Regression and Random Forest models

model_comparison_data = [
    ("Linear Regression", float(lr_rmse), float(lr_mae), float(lr_r2)),
    ("Random Forest Regression", float(rf_rmse), float(rf_mae), float(rf_r2))
]

model_comparison_df = spark.createDataFrame(
    model_comparison_data,
    ["model_name", "rmse", "mae", "r2"]
)

display(model_comparison_df)

In [0]:
# Select best model based on R2 score

best_model_name = "Linear Regression" if lr_r2 > rf_r2 else "Random Forest Regression"

print("Best model:", best_model_name)

In [0]:
# Step 11: Create final ML predictions using the best model

final_predictions_df = lr_model.transform(ml_vector_df)

display(
    final_predictions_df.select(
        "gnd_uid",
        "province_name",
        "district_name",
        "ds_division_name",
        "gn_division_name",
        target_col,
        "lr_predicted_opportunity_score",
        "opportunity_tier"
    ).limit(20)
)

In [0]:
# Rename Linear Regression prediction column

from pyspark.sql.functions import col, round as spark_round, abs as spark_abs

ml_predictions_df = final_predictions_df.withColumnRenamed(
    "lr_predicted_opportunity_score",
    "predicted_opportunity_score"
)

In [0]:
# Validate final prediction output

print("Final prediction rows:", ml_predictions_df.count())

ml_predictions_df.select(
    target_col,
    "predicted_opportunity_score",
    "opportunity_tier"
).printSchema()

In [0]:
# Step 12: Add prediction error and ML prediction tier

from pyspark.sql.functions import col, abs as spark_abs, round as spark_round, when

ml_predictions_enriched_df = ml_predictions_df.withColumn(
    "prediction_error",
    spark_round(
        spark_abs(col(target_col) - col("predicted_opportunity_score")),
        4
    )
).withColumn(
    "predicted_opportunity_score",
    spark_round(col("predicted_opportunity_score"), 2)
).withColumn(
    "actual_opportunity_score",
    spark_round(col(target_col), 2)
).withColumn(
    "ml_prediction_tier",
    when(col("predicted_opportunity_score") >= 80, "Very High")
    .when(col("predicted_opportunity_score") >= 60, "High")
    .when(col("predicted_opportunity_score") >= 40, "Medium")
    .otherwise("Low")
)

display(
    ml_predictions_enriched_df.select(
        "gnd_uid",
        "province_name",
        "district_name",
        "ds_division_name",
        "gn_division_name",
        "actual_opportunity_score",
        "predicted_opportunity_score",
        "prediction_error",
        "opportunity_tier",
        "ml_prediction_tier"
    ).limit(20)
)

In [0]:
# Validate ML prediction tier distribution

display(
    ml_predictions_enriched_df.groupBy("ml_prediction_tier")
    .count()
    .orderBy("ml_prediction_tier")
)

print("ML enriched rows:", ml_predictions_enriched_df.count())

In [0]:
# Step 12B: Create percentile-based ML prediction tier

from pyspark.sql.window import Window
from pyspark.sql.functions import percent_rank

ml_rank_window = Window.orderBy(col("predicted_opportunity_score"))

ml_predictions_enriched_df = ml_predictions_enriched_df.withColumn(
    "ml_prediction_percentile",
    percent_rank().over(ml_rank_window)
).withColumn(
    "ml_prediction_tier",
    when(col("ml_prediction_percentile") >= 0.90, "Very High")
    .when(col("ml_prediction_percentile") >= 0.70, "High")
    .when(col("ml_prediction_percentile") >= 0.30, "Medium")
    .otherwise("Low")
)

display(
    ml_predictions_enriched_df.groupBy("ml_prediction_tier")
    .count()
    .orderBy("ml_prediction_tier")
)

In [0]:
# Fixed Recovery Cell: Rebuild ML prediction dataframe from Gold table

from pyspark.sql.functions import col, abs as spark_abs, round as spark_round, when
from pyspark.sql.window import Window
from pyspark.sql.functions import percent_rank
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# 1. Load Gold scored table
gold_table = "workspace.census360.gold_gn_telco_scored_v2"
df = spark.table(gold_table)

# 2. Define feature columns and target column
feature_cols = [
    "population_total",
    "occupied_housing_units",
    "household_size",
    "male_share",
    "female_share",
    "child_share",
    "working_age_share",
    "elderly_share",
    "population_score",
    "housing_score",
    "household_size_score",
    "child_share_score",
    "working_age_score"
]

target_col = "telco_demand_opportunity_score"

# 3. Select ML columns WITHOUT duplicates
ml_df = df.select(
    "gnd_uid",
    "province_name",
    "district_name",
    "ds_division_name",
    "gn_division_name",
    *feature_cols,
    "mobile_data_demand_score",
    "home_broadband_opportunity_score",
    "youth_package_opportunity_score",
    "rural_coverage_priority_score",
    target_col,
    "opportunity_tier"
)

# 4. Fill nulls
numeric_cols = feature_cols + [
    target_col,
    "mobile_data_demand_score",
    "home_broadband_opportunity_score",
    "youth_package_opportunity_score",
    "rural_coverage_priority_score"
]

ml_clean_df = ml_df.fillna(0, subset=numeric_cols)

# 5. Assemble features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

ml_vector_df = assembler.transform(ml_clean_df)

# 6. Train Linear Regression again
train_df, test_df = ml_vector_df.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(
    featuresCol="features",
    labelCol=target_col,
    predictionCol="predicted_opportunity_score"
)

lr_model = lr.fit(train_df)

# 7. Predict all rows
ml_predictions_df = lr_model.transform(ml_vector_df)

# 8. Add error, actual score, percentile, and ML tier
ml_predictions_enriched_df = ml_predictions_df.withColumn(
    "prediction_error",
    spark_round(
        spark_abs(col(target_col) - col("predicted_opportunity_score")),
        4
    )
).withColumn(
    "predicted_opportunity_score",
    spark_round(col("predicted_opportunity_score"), 2)
).withColumn(
    "actual_opportunity_score",
    spark_round(col(target_col), 2)
)

ml_rank_window = Window.orderBy(col("predicted_opportunity_score"))

ml_predictions_enriched_df = ml_predictions_enriched_df.withColumn(
    "ml_prediction_percentile",
    percent_rank().over(ml_rank_window)
).withColumn(
    "ml_prediction_tier",
    when(col("ml_prediction_percentile") >= 0.90, "Very High")
    .when(col("ml_prediction_percentile") >= 0.70, "High")
    .when(col("ml_prediction_percentile") >= 0.30, "Medium")
    .otherwise("Low")
)

# 9. Create final output dataframe
final_ml_output_df = ml_predictions_enriched_df.select(
    "gnd_uid",
    "province_name",
    "district_name",
    "ds_division_name",
    "gn_division_name",
    "population_total",
    "occupied_housing_units",
    "household_size",
    "mobile_data_demand_score",
    "home_broadband_opportunity_score",
    "youth_package_opportunity_score",
    "rural_coverage_priority_score",
    "actual_opportunity_score",
    "predicted_opportunity_score",
    "prediction_error",
    "opportunity_tier",
    "ml_prediction_percentile",
    "ml_prediction_tier"
)

print("Final ML output rows:", final_ml_output_df.count())
print("Final ML output columns:", len(final_ml_output_df.columns))

display(final_ml_output_df.limit(10))

In [0]:
final_ml_output_df.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.census360.gold_ml_opportunity_predictions"
)

print("Saved table: workspace.census360.gold_ml_opportunity_predictions")

In [0]:
saved_ml_df = spark.table("workspace.census360.gold_ml_opportunity_predictions")

print("Saved ML rows:", saved_ml_df.count())
print("Saved ML columns:", len(saved_ml_df.columns))

display(
    saved_ml_df.groupBy("ml_prediction_tier")
    .count()
    .orderBy("ml_prediction_tier")
)

In [0]:
# Step 14: Create model comparison table using saved metric values

model_comparison_data = [
    ("Linear Regression", 0.0033996206638594274, 0.002842849278052953, 0.9999986753094794, "Best Model"),
    ("Random Forest Regression", 1.0181564702796897, 0.2288729457210453, 0.8811819121752403, "Compared Model")
]

model_comparison_df = spark.createDataFrame(
    model_comparison_data,
    ["model_name", "rmse", "mae", "r2", "model_status"]
)

display(model_comparison_df)

In [0]:
# Save model comparison table

model_comparison_df.write.mode("overwrite").format("delta").saveAsTable(
    "workspace.census360.gold_ml_model_comparison"
)

print("Saved table: workspace.census360.gold_ml_model_comparison")

In [0]:
# Validate model comparison table

saved_model_comparison_df = spark.table("workspace.census360.gold_ml_model_comparison")

print("Model comparison rows:", saved_model_comparison_df.count())

display(saved_model_comparison_df)

In [0]:
# Step 15: Final validation of ML output tables

tables_to_validate = [
    "workspace.census360.gold_ml_opportunity_predictions",
    "workspace.census360.gold_ml_model_comparison"
]

for table_name in tables_to_validate:
    table_df = spark.table(table_name)
    print(table_name, ":", table_df.count(), "rows,", len(table_df.columns), "columns")

In [0]:
import mlflow

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())

print("\nSpark ML objects currently available:")

found = False

for variable_name, variable_value in sorted(globals().items()):
    object_type = type(variable_value)
    object_module = getattr(object_type, "__module__", "")

    if object_module.startswith("pyspark.ml"):
        print(
            f"{variable_name} -> "
            f"{object_module}.{object_type.__name__}"
        )
        found = True

if not found:
    print(
        "No Spark ML objects are currently loaded. "
        "The model-training cells may need to be rerun."
    )

In [0]:
from pyspark.sql import DataFrame

# Inspect the feature assembler
feature_columns = assembler.getInputCols()
assembled_column = assembler.getOutputCol()

# Inspect the trained Linear Regression model
label_column = lr_model.getLabelCol()
model_features_column = lr_model.getFeaturesCol()
prediction_column = lr_model.getPredictionCol()

print("ASSEMBLER CONFIGURATION")
print("Input columns:", feature_columns)
print("Output column:", assembled_column)
print("Handle invalid:", assembler.getHandleInvalid())

print("\nLINEAR REGRESSION CONFIGURATION")
print("Label column:", label_column)
print("Features column:", model_features_column)
print("Prediction column:", prediction_column)

print("\nPOSSIBLE SPARK DATAFRAMES")

found_dataframe = False

for variable_name, variable_value in sorted(globals().items()):

    if isinstance(variable_value, DataFrame):

        dataframe_columns = variable_value.columns

        raw_features_available = all(
            column in dataframe_columns
            for column in feature_columns
        )

        print(f"\n{variable_name}")
        print(f"  Number of columns: {len(dataframe_columns)}")
        print(f"  Has all raw feature columns: {raw_features_available}")
        print(f"  Has label column: {label_column in dataframe_columns}")
        print(f"  Has assembled features: {assembled_column in dataframe_columns}")
        print(f"  Has prediction: {prediction_column in dataframe_columns}")

        found_dataframe = True

if not found_dataframe:
    print("No Spark DataFrames are currently available.")

In [0]:
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F

# Combine raw-feature preparation and the trained Linear Regression model
telco_lr_pipeline_model = PipelineModel(
    stages=[
        assembler,
        lr_model
    ]
)

# Create a small input dataset containing only the raw ML features
pipeline_test_input_df = (
    df
    .select(*feature_columns)
    .limit(5)
)

# Test the complete pipeline
pipeline_test_predictions_df = (
    telco_lr_pipeline_model
    .transform(pipeline_test_input_df)
)

prediction_column = lr_model.getPredictionCol()

# Validate the pipeline
test_row_count = pipeline_test_predictions_df.count()

null_prediction_count = (
    pipeline_test_predictions_df
    .filter(F.col(prediction_column).isNull())
    .count()
)

print("PIPELINE VALIDATION")
print("Pipeline variable: telco_lr_pipeline_model")
print("Pipeline stages:", [
    type(stage).__name__
    for stage in telco_lr_pipeline_model.stages
])
print("Test rows:", test_row_count)
print("Prediction column:", prediction_column)
print(
    "Prediction column exists:",
    prediction_column in pipeline_test_predictions_df.columns
)
print("Null predictions:", null_prediction_count)

print("\nSAMPLE PREDICTIONS")

pipeline_test_predictions_df.select(
    "population_total",
    "occupied_housing_units",
    "household_size",
    "child_share",
    "working_age_share",
    prediction_column
).show(
    5,
    truncate=False
)

In [0]:
from pyspark.sql import functions as F

# Replace both NULL and NaN feature values with 0.0.
# This matches the missing-value treatment used during model preparation.
clean_feature_expressions = []

for column_name in feature_columns:
    numeric_column = F.col(column_name).cast("double")

    clean_feature_expressions.append(
        F.when(
            numeric_column.isNull() | F.isnan(numeric_column),
            F.lit(0.0)
        )
        .otherwise(numeric_column)
        .alias(column_name)
    )

clean_pipeline_test_input_df = (
    df
    .select(*clean_feature_expressions)
    .limit(5)
)

clean_pipeline_test_predictions_df = (
    telco_lr_pipeline_model
    .transform(clean_pipeline_test_input_df)
)

prediction_column = lr_model.getPredictionCol()

invalid_prediction_count = (
    clean_pipeline_test_predictions_df
    .filter(
        F.col(prediction_column).isNull()
        | F.isnan(F.col(prediction_column))
    )
    .count()
)

print("CLEAN PIPELINE VALIDATION")
print("Test rows:", clean_pipeline_test_predictions_df.count())
print("Prediction column:", prediction_column)
print("Invalid NULL or NaN predictions:", invalid_prediction_count)

print("\nCLEAN SAMPLE PREDICTIONS")

clean_pipeline_test_predictions_df.select(
    "population_total",
    "occupied_housing_units",
    "household_size",
    "child_share",
    "working_age_share",
    prediction_column
).show(
    5,
    truncate=False
)

In [0]:
from pyspark.ml import PipelineModel
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.sql import functions as F

# Create separate cleaned feature names
cleaned_feature_columns = [
    f"clean_{column_name}"
    for column_name in feature_columns
]

# Build SQL expressions that convert:
# NULL -> 0.0
# NaN  -> 0.0
cleaning_expressions = []

for source_column, cleaned_column in zip(
    feature_columns,
    cleaned_feature_columns
):
    escaped_column = source_column.replace("`", "``")

    cleaning_expressions.append(
        f"""
        COALESCE(
            NANVL(
                CAST(`{escaped_column}` AS DOUBLE),
                CAST(0.0 AS DOUBLE)
            ),
            CAST(0.0 AS DOUBLE)
        ) AS `{cleaned_column}`
        """
    )

cleaning_statement = (
    "SELECT *, "
    + ", ".join(cleaning_expressions)
    + " FROM __THIS__"
)

# Stage 1: Clean raw feature values
feature_cleaner = SQLTransformer(
    statement=cleaning_statement
)

# Stage 2: Assemble cleaned values into the features vector
production_assembler = VectorAssembler(
    inputCols=cleaned_feature_columns,
    outputCol="features",
    handleInvalid="keep"
)

# Stage 3: Use the already-trained Linear Regression model
registered_telco_pipeline_model = PipelineModel(
    stages=[
        feature_cleaner,
        production_assembler,
        lr_model
    ]
)

# Include both problematic rows and valid high-population rows
problem_test_rows_df = (
    df
    .select(*feature_columns)
    .limit(5)
)

valid_test_rows_df = (
    df
    .select(*feature_columns)
    .filter(
        F.col("population_total").cast("double") > 0
    )
    .orderBy(
        F.col("population_total").desc_nulls_last()
    )
    .limit(5)
)

registration_test_input_df = (
    problem_test_rows_df
    .unionByName(valid_test_rows_df)
)

registration_test_predictions_df = (
    registered_telco_pipeline_model
    .transform(registration_test_input_df)
)

invalid_prediction_count = (
    registration_test_predictions_df
    .filter(
        F.col("predicted_opportunity_score").isNull()
        | F.isnan(F.col("predicted_opportunity_score"))
    )
    .count()
)

print("FINAL REGISTRATION PIPELINE VALIDATION")
print(
    "Pipeline stages:",
    [
        type(stage).__name__
        for stage in registered_telco_pipeline_model.stages
    ]
)
print(
    "Test rows:",
    registration_test_predictions_df.count()
)
print(
    "Invalid NULL or NaN predictions:",
    invalid_prediction_count
)

registration_test_predictions_df.select(
    "population_total",
    "occupied_housing_units",
    "household_size",
    "child_share",
    "working_age_share",
    "predicted_opportunity_score"
).show(
    10,
    truncate=False
)

In [0]:
import math
import mlflow
import mlflow.spark

from mlflow import MlflowClient
from mlflow.models import infer_signature
from pyspark.ml import PipelineModel
from pyspark.ml.feature import SQLTransformer
from pyspark.sql import functions as F


# ---------------------------------------------------------
# 1. Unity Catalog model configuration
# ---------------------------------------------------------

mlflow.set_registry_uri("databricks-uc")

registered_model_name = (
    "workspace.census360."
    "census360_telco_opportunity_model"
)

print("Registry URI:", mlflow.get_registry_uri())
print("Registered model name:", registered_model_name)


# ---------------------------------------------------------
# 2. Add MLflow-compatible output column
# ---------------------------------------------------------

prediction_output_transformer = SQLTransformer(
    statement="""
        SELECT *,
               CAST(
                   `predicted_opportunity_score`
                   AS DOUBLE
               ) AS `prediction`
        FROM __THIS__
    """
)

mlflow_ready_telco_model = PipelineModel(
    stages=[
        feature_cleaner,
        production_assembler,
        lr_model,
        prediction_output_transformer
    ]
)


# ---------------------------------------------------------
# 3. Create signature and input example
# ---------------------------------------------------------

signature_input_df = (
    df
    .select(
        *[
            F.col(column_name)
            .cast("double")
            .alias(column_name)
            for column_name in feature_columns
        ]
    )
    .filter(
        F.col("population_total").isNotNull()
        & (F.col("population_total") > 0)
    )
    .limit(20)
)

signature_prediction_df = (
    mlflow_ready_telco_model
    .transform(signature_input_df)
    .select(
        F.col("prediction").cast("double")
    )
)

signature = infer_signature(
    signature_input_df,
    signature_prediction_df
)

input_example = (
    signature_input_df
    .limit(5)
    .toPandas()
)

invalid_prediction_count = (
    signature_prediction_df
    .filter(
        F.col("prediction").isNull()
        | F.isnan(F.col("prediction"))
    )
    .count()
)

print("\nPRE-REGISTRATION CHECK")
print("Signature input rows:", signature_input_df.count())
print("Invalid predictions:", invalid_prediction_count)
print("Input columns:", len(feature_columns))
print("Output column: prediction")

if invalid_prediction_count != 0:
    raise ValueError(
        "Registration stopped because invalid predictions were found."
    )


# ---------------------------------------------------------
# 4. Close any leftover active MLflow run
# ---------------------------------------------------------

if mlflow.active_run() is not None:
    mlflow.end_run()


# ---------------------------------------------------------
# 5. Log metrics, model, signature and input example
# ---------------------------------------------------------

with mlflow.start_run(
    run_name="census360_linear_regression_registration"
) as active_run:

    mlflow.log_param(
        "model_type",
        "Spark Linear Regression Pipeline"
    )

    mlflow.log_param(
        "target_column",
        "telco_demand_opportunity_score"
    )

    mlflow.log_param(
        "feature_count",
        len(feature_columns)
    )

    mlflow.log_param(
        "missing_value_strategy",
        "NULL and NaN converted to 0.0"
    )

    mlflow.log_param(
        "pipeline_stages",
        "SQLTransformer, VectorAssembler, "
        "LinearRegressionModel, SQLTransformer"
    )

    # Log evaluation metrics if these variables are available
    metric_variables = {
        "linear_regression_rmse": globals().get("lr_rmse"),
        "linear_regression_mae": globals().get("lr_mae"),
        "linear_regression_r2": globals().get("lr_r2")
    }

    for metric_name, metric_value in metric_variables.items():

        if metric_value is not None:

            metric_value = float(metric_value)

            if math.isfinite(metric_value):
                mlflow.log_metric(
                    metric_name,
                    metric_value
                )

    model_info = mlflow.spark.log_model(
    spark_model=mlflow_ready_telco_model,
    artifact_path="model",
    registered_model_name=registered_model_name,
    signature=signature,
    input_example=input_example,

    # Required temporary UC Volume path for Free Edition serverless
    dfs_tmpdir=mlflow_tmp_dir,

    await_registration_for=300,
    metadata={
        "project": "Census360 Sri Lanka",
        "business_use": (
            "GN-level telco demand opportunity prediction"
        ),
        "best_model": "Linear Regression"
    }
)

    current_run_id = active_run.info.run_id


# ---------------------------------------------------------
# 6. Verify the registered model version
# ---------------------------------------------------------

client = MlflowClient()

model_versions = list(
    client.search_model_versions(
        f"name='{registered_model_name}'"
    )
)

if not model_versions:
    raise RuntimeError(
        "No registered model versions were found."
    )

latest_version = max(
    model_versions,
    key=lambda version: int(version.version)
)

print("\nMODEL REGISTRATION COMPLETED")
print("Run ID:", current_run_id)
print("Logged model URI:", model_info.model_uri)
print("Registered model:", latest_version.name)
print("Latest version:", latest_version.version)
print("Registration status:", latest_version.status)

In [0]:
# Existing Unity Catalog Volume — no paid resource is created
mlflow_tmp_dir = (
    "/Volumes/workspace/default/"
    "census360_raw/_mlflow_tmp"
)

# Create the temporary subfolder if it does not already exist
dbutils.fs.mkdirs(mlflow_tmp_dir)

print("MLflow temporary directory prepared:")
print(mlflow_tmp_dir)

print("\nCurrent folder contents:")
for item in dbutils.fs.ls(mlflow_tmp_dir):
    print(item.path)